# Quality再検証・実行環境の修正

歴史的な実験コードです。現在の実行入口は `../09_confidence_nested.ipynb`。
元Notebookのセル番号は0始まりです。コードの個人フォルダ名は置換しています。独立実行は保証しません。
保存出力は `../../results/imported_20260907/`、監査は `../../docs/CONFIDENCE_AUDIT.md` を参照してください。


## 元のセル index 16


In [ ]:
# -*- coding: utf-8 -*-
"""USD/JPY: 次に実行するBASE対QUALITY検証（1ファイル版）。

最初に一度インストール:
    python -m pip install numpy pandas scikit-learn matplotlib yfinance
実行（新しく取得する場合）:
    python fx_next_experiment.py
固定CSVを再使用:
    python fx_next_experiment.py --csv "usdjpy_5m.csv"
Jupyterでは別セルで:
    %run fx_next_experiment.py

GitHubの修正済みロジックを1ファイル化した診断用スクリプトです。
実データ・他セルのdf・fx_researchパッケージには依存しません。
CSV指定がない場合のみYahoo Financeから直近59日を取得して固定保存します。
公式仕様: https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html

【何を確認するか】
1. BASE / QUALITYの未見期間の平均純損益・PF・DD・取引数。
2. Qualityスコアが高いBASE取引ほど実損益が良いか（事後診断のみ）。
3. BUY / SELL、foldごとの違い。
4. 同じ取引のコストだけ1.5倍・2倍にしたときの損益感度。

【ルール】
シグナル足確定 → 次足Open → 最大6本目Close。同一足TP/SLはSL先行。
SELLはエントリー元本基準。初期equity=1を含めてDDを計算。
Qualityの正解は「方向付き時間決済リターン－コスト > 0」を維持。
Testは設定選択に使わず、今回出力するスコア帯もTestで調整しません。
結果ZIPには生のOHLC CSVを入れません。生データは手元に保存します。
"""
from __future__ import annotations


# ====================================================================
# 1. 固定設定（Testの結果に合わせて変更しない）
# ====================================================================

HORIZON_BARS = 6
MOVE_THRESHOLD = 0.0005
TRADING_COST = 1.33e-05
HALF_LIFE_DAYS = 20
OUTER_SPLITS = 5
QUALITY_OOF_SPLITS = 4
MIN_VALIDATION_TRADES = 15
MOVE_PROB_LIST = [0.55, 0.6, 0.65, 0.7]
DIRECTION_PROB_LIST = [0.55, 0.6, 0.65, 0.7]
QUALITY_PROB_LIST = [0.5, 0.55, 0.6, 0.65, 0.7]
TP_LIST = [0.0005, 0.0008, 0.001]
SL_LIST = [0.0005, 0.0007, 0.001]


# ====================================================================
# 2. モデルに渡す特徴量の一覧
# ====================================================================

move_features = [
    "volatility_1h",
    "volatility_2h",
    "volatility_4h",
    "range",
    "body",
    "return_5m",
    "return_15m",
    "return_30m",
    "MA20_slope",
    "MA50_slope",
    "distance_high_1h",
    "distance_low_1h",
    "hour_sin",
    "hour_cos",
    "weekday",
]

direction_features = [
    "return_5m",
    "return_15m",
    "return_30m",
    "return_1h",
    "return_2h",
    "MA5_distance",
    "MA20_distance",
    "MA50_distance",
    "MA5_slope",
    "MA20_slope",
    "MA50_slope",
    "RSI",
    "bullish",
    "body",
    "upper_wick",
    "lower_wick",
    "distance_high_1h",
    "distance_low_1h",
    "volatility_1h",
    "hour_sin",
    "hour_cos",
    "weekday",
]

quality_market_features = [
    "return_5m",
    "return_15m",
    "return_30m",
    "return_1h",
    "return_2h",
    "volatility_1h",
    "volatility_2h",
    "volatility_4h",
    "ATR14_pct",
    "ADX14",
    "RSI",
    "MA5_distance",
    "MA20_distance",
    "MA50_distance",
    "MA5_slope",
    "MA20_slope",
    "MA50_slope",
    "distance_high_1h",
    "distance_low_1h",
    "body",
    "range",
    "upper_wick",
    "lower_wick",
    "hour_sin",
    "hour_cos",
    "weekday",
]


# ====================================================================
# 3. CSVの検査
# ====================================================================

from pathlib import Path

import numpy as np
import pandas as pd


def load_bars(path: str | Path) -> pd.DataFrame:
    """Read bar-open timestamps with explicit offsets and normalize to Japan time."""
    frame = pd.read_csv(path)
    required = ["timestamp", "Open", "High", "Low", "Close"]
    if not set(required).issubset(frame.columns):
        raise ValueError(f"CSV must contain {required}")
    timestamps = frame.pop("timestamp").astype(str)
    if not timestamps.str.contains(r"(?:Z|[+-]\d{2}:?\d{2})$", regex=True).all():
        raise ValueError(
            "Every timestamp must include Z or a UTC offset; naive times are ambiguous."
        )
    frame.index = pd.DatetimeIndex(pd.to_datetime(timestamps, utc=True)).tz_convert(
        "Asia/Tokyo"
    )
    frame = frame[["Open", "High", "Low", "Close"]].apply(pd.to_numeric, errors="raise")
    if (
        frame.empty
        or not frame.index.is_monotonic_increasing
        or not frame.index.is_unique
    ):
        raise ValueError(
            "Bars must be nonempty, strictly chronological and unique; input is not silently sorted."
        )
    if not np.isfinite(frame.to_numpy()).all() or (frame <= 0).any().any():
        raise ValueError("OHLC must be finite and positive.")
    if (frame.High < frame[["Open", "Close", "Low"]].max(axis=1)).any():
        raise ValueError("High is inconsistent with OHLC.")
    if (frame.Low > frame[["Open", "Close", "High"]].min(axis=1)).any():
        raise ValueError("Low is inconsistent with OHLC.")
    if (
        (frame.index.minute % 5 != 0)
        | (frame.index.second != 0)
        | (frame.index.microsecond != 0)
        | (frame.index.nanosecond != 0)
    ).any():
        raise ValueError("Bar timestamps must align to a five-minute grid.")
    return frame


# ====================================================================
# 4. 過去だけから特徴量を作る
# ====================================================================

import numpy as np
import pandas as pd


def make_features(bars):
    """Use only prices through the signal bar; keep the original row index."""
    df = bars.copy()
    df["return_5m"] = df["Close"].pct_change(1)
    df["return_15m"] = df["Close"].pct_change(3)
    df["return_30m"] = df["Close"].pct_change(6)
    df["return_1h"] = df["Close"].pct_change(12)
    df["return_2h"] = df["Close"].pct_change(24)
    df["MA5"] = df["Close"].rolling(5).mean()
    df["MA20"] = df["Close"].rolling(20).mean()
    df["MA50"] = df["Close"].rolling(50).mean()
    df["MA5_distance"] = df["Close"] / df["MA5"] - 1
    df["MA20_distance"] = df["Close"] / df["MA20"] - 1
    df["MA50_distance"] = df["Close"] / df["MA50"] - 1
    df["MA5_slope"] = df["MA5"].pct_change(3)
    df["MA20_slope"] = df["MA20"].pct_change(3)
    df["MA50_slope"] = df["MA50"].pct_change(3)
    df["body"] = abs(df["Close"] - df["Open"]) / df["Open"]
    df["range"] = (df["High"] - df["Low"]) / df["Close"]
    df["upper_wick"] = (df["High"] - df[["Open", "Close"]].max(axis=1)) / df["Close"]
    df["lower_wick"] = (df[["Open", "Close"]].min(axis=1) - df["Low"]) / df["Close"]
    df["bullish"] = (df["Close"] > df["Open"]).astype(int)
    df["volatility_1h"] = df["return_5m"].rolling(12).std()
    df["volatility_2h"] = df["return_5m"].rolling(24).std()
    df["volatility_4h"] = df["return_5m"].rolling(48).std()
    delta = df["Close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()
    rs = avg_gain / avg_loss
    df["RSI"] = 100 - 100 / (1 + rs)
    df["high_1h"] = df["High"].rolling(12).max()
    df["low_1h"] = df["Low"].rolling(12).min()
    df["distance_high_1h"] = df["Close"] / df["high_1h"] - 1
    df["distance_low_1h"] = df["Close"] / df["low_1h"] - 1
    df["hour"] = df.index.hour
    df["weekday"] = df.index.dayofweek
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    previous_close = df["Close"].shift(1)
    tr1 = df["High"] - df["Low"]
    tr2 = abs(df["High"] - previous_close)
    tr3 = abs(df["Low"] - previous_close)
    true_range = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    df["ATR14"] = true_range.rolling(14).mean()
    df["ATR14_pct"] = df["ATR14"] / df["Close"]
    high_diff = df["High"].diff()
    low_diff = -df["Low"].diff()
    plus_dm = np.where((high_diff > low_diff) & (high_diff > 0), high_diff, 0.0)
    minus_dm = np.where((low_diff > high_diff) & (low_diff > 0), low_diff, 0.0)
    plus_dm = pd.Series(plus_dm, index=df.index)
    minus_dm = pd.Series(minus_dm, index=df.index)
    atr_adx = true_range.rolling(14).mean()
    plus_di = 100 * plus_dm.rolling(14).mean() / atr_adx
    minus_di = 100 * minus_dm.rolling(14).mean() / atr_adx
    dx = 100 * abs(plus_di - minus_di) / (plus_di + minus_di)
    df["ADX14"] = dx.rolling(14).mean()
    return df.replace([np.inf, -np.inf], np.nan)


# ====================================================================
# 5. 未来の正解ラベルを作る
# ====================================================================

import numpy as np
import pandas as pd



def prepare_data(bars):
    """Build labels on original bar positions before dropping unavailable features.

    Timestamp convention: each input index denotes the bar's opening time.
    A label uses Open(t+1) through Close(t+6), available at index[t+6]+5min.
    Price-path gaps across the horizon are excluded, never filled.
    """
    frame = make_features(bars)
    frame["bar_position"] = np.arange(len(bars))
    frame["entry_price"] = bars.Open.shift(-1)
    frame["exit_price"] = bars.Close.shift(-HORIZON_BARS)
    frame["future_return"] = frame.exit_price / frame.entry_price - 1
    times = pd.Series(bars.index, index=bars.index)
    frame["label_end"] = times.shift(-HORIZON_BARS) + pd.Timedelta(minutes=5)
    complete = (times.shift(-HORIZON_BARS) - times).eq(
        pd.Timedelta(minutes=5 * HORIZON_BARS)
    )
    frame["move_target"] = frame.future_return.abs().gt(MOVE_THRESHOLD).astype(int)
    frame["direction_target"] = frame.future_return.gt(0).astype(int)
    required = list(
        dict.fromkeys(move_features + direction_features + quality_market_features)
    )
    required += ["future_return", "entry_price", "exit_price", "label_end"]
    return frame.loc[complete].dropna(subset=required).copy()


# ====================================================================
# 6. 時間順に学習・検証・Testを分ける
# ====================================================================

from dataclasses import dataclass

import pandas as pd



@dataclass(frozen=True)
class Fold:
    number: int
    train: pd.DataFrame
    core: pd.DataFrame
    validation: pd.DataFrame
    test: pd.DataFrame


def outer_folds(data):
    block = len(data) // (OUTER_SPLITS + 1)
    if block < 1:
        return
    for number in range(1, OUTER_SPLITS + 1):
        train_end = block * number
        test_start = train_end + HORIZON_BARS
        test_end = min(test_start + block, len(data))
        train = data.iloc[:train_end]
        cut = int(len(train) * 0.8)
        core = train.iloc[: max(0, cut - HORIZON_BARS)]
        validation = train.iloc[cut:]
        test = data.iloc[test_start:test_end]
        # Bar-open cutoffs are deliberately conservative: all outcomes are known before the next block.
        if len(validation) and len(core):
            if not (core.label_end <= validation.index[0]).all():
                raise ValueError("Training label crosses validation boundary.")
        if len(test) and len(train):
            if not (train.label_end <= test.index[0]).all():
                raise ValueError("Training label crosses test boundary.")
        yield Fold(number, train, core, validation, test)


# ====================================================================
# 7. MOVEとDirectionを学習する
# ====================================================================

import numpy as np
from sklearn.ensemble import RandomForestClassifier


def make_time_weights(index, half_life_days):
    latest = index.max()
    age_days = (latest - index).total_seconds() / 86400
    weights = 0.5 ** (age_days / half_life_days)
    return np.array(weights)


def fit_base_models(train_frame, trees=250):
    if train_frame["move_target"].nunique() < 2:
        return None
    move_weights = make_time_weights(train_frame.index, HALF_LIFE_DAYS)
    move_model = RandomForestClassifier(
        n_estimators=trees,
        max_depth=8,
        min_samples_leaf=20,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    move_model.fit(
        train_frame[move_features],
        train_frame["move_target"],
        sample_weight=move_weights,
    )
    direction_train = train_frame[train_frame["move_target"] == 1]
    if len(direction_train) < 50 or direction_train["direction_target"].nunique() < 2:
        return None
    direction_weights = make_time_weights(direction_train.index, HALF_LIFE_DAYS)
    direction_model = RandomForestClassifier(
        n_estimators=trees,
        max_depth=8,
        min_samples_leaf=15,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    direction_model.fit(
        direction_train[direction_features],
        direction_train["direction_target"],
        sample_weight=direction_weights,
    )
    return (move_model, direction_model)


def predict_base_models(models, frame):
    move_model, direction_model = models
    p_move = move_model.predict_proba(frame[move_features])[:, 1]
    direction_prob = direction_model.predict_proba(frame[direction_features])
    class_map = {c: i for i, c in enumerate(direction_model.classes_)}
    p_down = direction_prob[:, class_map[0]]
    p_up = direction_prob[:, class_map[1]]
    return (p_move, p_up, p_down)


# ====================================================================
# 8. 過去モデルの将来予測でQualityを学習する
# ====================================================================

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier


def build_quality_features(frame, p_move, p_up, p_down):
    quality_x = frame[quality_market_features].copy()
    quality_x["p_move"] = p_move
    quality_x["p_up"] = p_up
    quality_x["p_down"] = p_down
    quality_x["direction_confidence"] = abs(p_up - p_down)
    quality_x["predicted_direction"] = (p_up >= p_down).astype(int)
    return quality_x


def make_quality_target(frame, p_up, p_down):
    predicted_buy = p_up >= p_down
    directional_return = np.where(
        predicted_buy, frame["future_return"].values, -frame["future_return"].values
    )
    net_return = directional_return - TRADING_COST
    quality_target = (net_return > 0).astype(int)
    return (quality_target, net_return)


def create_quality_oof_dataset(frame):
    initial_size = int(len(frame) * 0.4)
    remaining = len(frame) - initial_size
    block = max(remaining // QUALITY_OOF_SPLITS, 1)
    qx_list = []
    for split in range(QUALITY_OOF_SPLITS):
        val_start = initial_size + split * block
        if split == QUALITY_OOF_SPLITS - 1:
            val_end = len(frame)
        else:
            val_end = min(val_start + block, len(frame))
        train_end = val_start - HORIZON_BARS
        if train_end < 200:
            continue
        oof_train = frame.iloc[:train_end]
        oof_val = frame.iloc[val_start:val_end]
        if len(oof_val) == 0:
            continue
        models = fit_base_models(oof_train, trees=180)
        if models is None:
            continue
        p_move, p_up, p_down = predict_base_models(models, oof_val)
        quality_x = build_quality_features(oof_val, p_move, p_up, p_down)
        quality_y, _ = make_quality_target(oof_val, p_up, p_down)
        quality_x["quality_target"] = quality_y
        qx_list.append(quality_x)
    if len(qx_list) == 0:
        return None
    quality_data = pd.concat(qx_list).sort_index()
    return quality_data


def fit_quality_model(quality_data):
    quality_feature_names = [c for c in quality_data.columns if c != "quality_target"]
    if len(quality_data) < 100 or quality_data["quality_target"].nunique() < 2:
        return None
    weights = make_time_weights(quality_data.index, HALF_LIFE_DAYS)
    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=7,
        min_samples_leaf=20,
        max_features="sqrt",
        class_weight="balanced",
        random_state=123,
        n_jobs=-1,
    )
    model.fit(
        quality_data[quality_feature_names],
        quality_data["quality_target"],
        sample_weight=weights,
    )
    return (model, quality_feature_names)


def predict_quality(quality_bundle, frame, p_move, p_up, p_down):
    model, feature_names = quality_bundle
    x = build_quality_features(frame, p_move, p_up, p_down)
    probability = model.predict_proba(x[feature_names])[:, 1]
    return probability


# ====================================================================
# 9. BUY / SELL / WAITへ変換する
# ====================================================================

import numpy as np


def make_signals(
    p_move, p_up, p_down, move_t, direction_t, quality_prob=None, quality_t=None
):
    buy = (p_move >= move_t) & (p_up >= direction_t) & (p_up > p_down)
    sell = (p_move >= move_t) & (p_down >= direction_t) & (p_down > p_up)
    if quality_prob is not None and quality_t is not None:
        quality_mask = quality_prob >= quality_t
        buy = buy & quality_mask
        sell = sell & quality_mask
    signals = np.zeros(len(p_move))
    signals[buy] = 1
    signals[sell] = -1
    return signals


# ====================================================================
# 10. 売買とコストを計算する
# ====================================================================

import numpy as np
import pandas as pd



def simulate_trade(
    bars, signal_time, direction, tp, sl, *, end_time=None, cost=TRADING_COST
):
    if direction not in ("BUY", "SELL"):
        raise ValueError("direction must be BUY or SELL")
    if not (
        np.isfinite([tp, sl, cost]).all() and 0 < tp < 1 and 0 < sl < 1 and cost >= 0
    ):
        raise ValueError(
            "Require 0 < TP/SL < 1 and finite nonnegative round-trip cost."
        )
    signal_loc = bars.index.get_loc(signal_time)
    final_loc = signal_loc + HORIZON_BARS
    if final_loc >= len(bars):
        return None
    final_time = bars.index[final_loc] + pd.Timedelta(minutes=5)
    if end_time is not None and final_time > end_time:
        return None
    if bars.index[final_loc] - bars.index[signal_loc] != pd.Timedelta(
        minutes=5 * HORIZON_BARS
    ):
        return None
    side = 1 if direction == "BUY" else -1
    entry = float(bars.iloc[signal_loc + 1].Open)
    exit_loc, exit_price, reason = final_loc, float(bars.iloc[final_loc].Close), "TIME"
    for loc in range(signal_loc + 1, final_loc + 1):
        bar = bars.iloc[loc]
        open_return = side * (float(bar.Open) / entry - 1)
        # An opening gap through a stop fills at the worse opening price.
        if open_return <= -sl:
            exit_loc, exit_price, reason = loc, float(bar.Open), "GAP_SL"
            break
        # Profit-taking at the limit price; do not grant favorable gap improvement.
        if open_return >= tp:
            exit_loc, exit_price, reason = loc, entry * (1 + side * tp), "TP"
            break
        favorable = side * (float(bar.High if side == 1 else bar.Low) / entry - 1)
        adverse = side * (float(bar.Low if side == 1 else bar.High) / entry - 1)
        if adverse <= -sl:
            exit_loc, exit_price, reason = loc, entry * (1 - side * sl), "SL"
            break
        if favorable >= tp:
            exit_loc, exit_price, reason = loc, entry * (1 + side * tp), "TP"
            break
    gross = side * (exit_price / entry - 1)
    # Bar-resolution bookkeeping time, not a claim about tick-level execution time.
    exit_time = bars.index[exit_loc] + pd.Timedelta(minutes=5)
    return {
        "signal_time": signal_time,
        "entry_time": bars.index[signal_loc + 1],
        "exit_time": exit_time,
        "direction": direction,
        "entry_price": entry,
        "exit_price": exit_price,
        "exit_reason": reason,
        "gross_return": gross,
        "cost": cost,
        "net_return": gross - cost,
        "signal_position": signal_loc,
    }


def run_backtest(bars, frame, signals, tp, sl, *, end_time=None, cost=TRADING_COST):
    if len(frame) != len(signals) or not np.isin(signals, [-1, 0, 1]).all():
        raise ValueError("Signals must align one-to-one with frame and be -1, 0 or 1.")
    records = []
    next_signal_position = -1
    for time, signal in zip(frame.index, signals):
        position = bars.index.get_loc(time)
        if signal == 0 or position < next_signal_position:
            continue
        trade = simulate_trade(
            bars,
            time,
            "BUY" if signal == 1 else "SELL",
            tp,
            sl,
            end_time=end_time,
            cost=cost,
        )
        if trade is not None:
            records.append(trade)
            # Even an early TP/SL retains the historical fixed-horizon lockout.
            next_signal_position = position + HORIZON_BARS
    columns = [
        "signal_time",
        "entry_time",
        "exit_time",
        "direction",
        "entry_price",
        "exit_price",
        "exit_reason",
        "gross_return",
        "cost",
        "net_return",
        "signal_position",
    ]
    return pd.DataFrame.from_records(records, columns=columns)


# ====================================================================
# 11. 平均損益・PF・ドローダウンを計算する
# ====================================================================

import numpy as np


def strategy_stats(returns):
    r = np.asarray(returns, dtype=float)
    if not np.isfinite(r).all() or (r <= -1).any():
        raise ValueError("Returns must be finite and greater than -100%.")
    if not len(r):
        return dict(
            trades=0,
            win_rate=np.nan,
            avg_return=np.nan,
            profit_factor=np.nan,
            max_dd=np.nan,
            total_growth=0.0,
        )
    gains, losses = r[r > 0].sum(), -r[r < 0].sum()
    pf = gains / losses if losses else (np.inf if gains else np.nan)
    equity = np.r_[1.0, np.cumprod(1 + r)]
    drawdown = equity / np.maximum.accumulate(equity) - 1
    return dict(
        trades=len(r),
        win_rate=float((r > 0).mean()),
        avg_return=float(r.mean()),
        profit_factor=float(pf),
        max_dd=float(drawdown.min()),
        total_growth=float(equity[-1] - 1),
    )


# ====================================================================
# 12. Validationで設定を選び、Testで比較する
# ====================================================================

import argparse
import hashlib
import importlib.metadata
import itertools
import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd



def run_experiment(csv_path, output_dir):
    """Persist settings, every fold status, trades, score diagnostics and input hash.

    Model formulae/forest sizes retain the notebook baseline. Corrected accounting,
    stricter boundaries and fixed data input make this a new experiment version.
    """
    csv_path, output_dir = Path(csv_path), Path(output_dir)
    bars = load_bars(csv_path)
    data = prepare_data(bars)
    if len(data) < 600:
        raise ValueError(
            "Need at least 600 usable rows; this is only a technical minimum, not statistical sufficiency."
        )
    output_dir.mkdir(parents=True, exist_ok=False)
    versions = {
        name: importlib.metadata.version(name)
        for name in ["numpy", "pandas", "scikit-learn"]
    }
    source_hashes = {Path(__file__).name: hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}
    metadata = {
        "status": "running",
        "experiment": "quality-v0.2-single-file-diagnostics",
        "input_filename": csv_path.name,
        "input_sha256": hashlib.sha256(csv_path.read_bytes()).hexdigest(),
        "bar_rows": len(bars),
        "usable_rows": len(data),
        "first_bar": str(bars.index[0]),
        "last_bar": str(bars.index[-1]),
        "timezone": "Asia/Tokyo",
        "python": platform.python_version(),
        "versions": versions,
        "source_sha256": source_hashes,
        "config": {'HORIZON_BARS': HORIZON_BARS, 'MOVE_THRESHOLD': MOVE_THRESHOLD, 'TRADING_COST': TRADING_COST, 'HALF_LIFE_DAYS': HALF_LIFE_DAYS, 'OUTER_SPLITS': OUTER_SPLITS, 'QUALITY_OOF_SPLITS': QUALITY_OOF_SPLITS, 'MIN_VALIDATION_TRADES': MIN_VALIDATION_TRADES, 'MOVE_PROB_LIST': MOVE_PROB_LIST, 'DIRECTION_PROB_LIST': DIRECTION_PROB_LIST, 'QUALITY_PROB_LIST': QUALITY_PROB_LIST, 'TP_LIST': TP_LIST, 'SL_LIST': SL_LIST},
        "notes": "No annualization. Closed-trade drawdown, no intratrade marking. Decimal return units.",
    }

    def save_metadata():
        (output_dir / "run.json").write_text(
            json.dumps(metadata, indent=2), encoding="utf-8"
        )

    save_metadata()
    rows, trade_frames, importance_frames, score_frames = [], [], [], []
    for fold in outer_folds(data):
        row = {
            "fold": fold.number,
            "status": "skipped",
            "reason": "",
            "train_rows": len(fold.train),
            "test_rows": len(fold.test),
            "test_start": str(fold.test.index[0]) if len(fold.test) else "",
            "test_end": str(fold.test.index[-1]) if len(fold.test) else "",
        }
        rows.append(row)
        print(f"Fold {fold.number}: preparing chronological training", flush=True)
        if len(fold.train) < 500 or not len(fold.test) or not len(fold.validation):
            row["reason"] = "insufficient_rows"
            continue
        print("  内部OOFとQuality学習中...", flush=True)
        oof = create_quality_oof_dataset(fold.core)
        quality = fit_quality_model(oof) if oof is not None else None
        base = fit_base_models(fold.core, trees=300)
        if quality is None or base is None:
            row["reason"] = "training_unavailable_or_one_class"
            continue
        probabilities = predict_base_models(base, fold.validation)
        quality_probabilities = predict_quality(
            quality, fold.validation, *probabilities
        )
        # Validation outcomes must be available before its boundary ends; original global df could look beyond it.
        val_end = fold.validation.index[-1] + pd.Timedelta(minutes=5)
        print("  Validationで閾値・TP/SLを選択中...", flush=True)
        best, best_score = None, -np.inf
        for move, direction, tp, sl in itertools.product(
            MOVE_PROB_LIST,
            DIRECTION_PROB_LIST,
            TP_LIST,
            SL_LIST,
        ):
            signals = make_signals(*probabilities, move, direction)
            trades = run_backtest(
                bars, fold.validation, signals, tp, sl, end_time=val_end
            )
            if len(trades) < MIN_VALIDATION_TRADES:
                continue
            score = trades.net_return.mean() * np.sqrt(len(trades))
            if score > best_score:
                best_score, best = score, (move, direction, tp, sl)
        if best is None:
            row["reason"] = "no_base_setting_meets_min_validation_trades"
            continue
        move, direction, tp, sl = best
        best_quality, quality_score = None, -np.inf
        for threshold in QUALITY_PROB_LIST:
            signals = make_signals(
                *probabilities,
                move,
                direction,
                quality_prob=quality_probabilities,
                quality_t=threshold,
            )
            trades = run_backtest(
                bars, fold.validation, signals, tp, sl, end_time=val_end
            )
            if len(trades) < MIN_VALIDATION_TRADES:
                continue
            score = trades.net_return.mean() * np.sqrt(len(trades))
            if score > quality_score:
                quality_score, best_quality = score, threshold
        row["quality_filter_disabled"] = best_quality is None
        threshold = best_quality if best_quality is not None else 0.0
        row.update(
            move_threshold=move,
            direction_threshold=direction,
            tp=tp,
            sl=sl,
            quality_threshold=threshold,
        )
        print("  設定を固定して再学習 → Test評価中...", flush=True)
        final_oof = create_quality_oof_dataset(fold.train)
        final_quality = fit_quality_model(final_oof) if final_oof is not None else None
        final_base = fit_base_models(fold.train, trees=400)
        if final_quality is None or final_base is None:
            row["reason"] = "final_retraining_unavailable"
            continue
        test_probabilities = predict_base_models(final_base, fold.test)
        test_quality = predict_quality(final_quality, fold.test, *test_probabilities)
        test_end = fold.test.index[-1] + pd.Timedelta(minutes=5)
        score_frames.append(
            pd.DataFrame(
                {
                    "fold": fold.number,
                    "signal_time": fold.test.index,
                    "p_move": test_probabilities[0],
                    "p_up": test_probabilities[1],
                    "p_down": test_probabilities[2],
                    "p_quality": test_quality,
                }
            )
        )
        for strategy, q in [("BASE", None), ("QUALITY", test_quality)]:
            signals = make_signals(
                *test_probabilities,
                move,
                direction,
                quality_prob=q,
                quality_t=threshold,
            )
            trades = run_backtest(bars, fold.test, signals, tp, sl, end_time=test_end)
            row.update(
                {
                    f"{strategy.lower()}_{k}": v
                    for k, v in strategy_stats(trades.net_return).items()
                }
            )
            trades["strategy"], trades["fold"] = strategy, fold.number
            trade_frames.append(trades)
        model, names = final_quality
        importance_frames.append(
            pd.DataFrame(
                {
                    "fold": fold.number,
                    "feature": names,
                    "importance": model.feature_importances_,
                }
            )
        )
        row["status"], row["reason"] = "evaluated", ""
        print(
            f"Fold {fold.number}: BASE {row['base_trades']}, QUALITY {row['quality_trades']} trades",
            flush=True,
        )
    pd.DataFrame(rows).to_csv(output_dir / "folds.csv", index=False)
    trades = (
        pd.concat(trade_frames, ignore_index=True)
        if trade_frames
        else pd.DataFrame(
            columns=["strategy", "fold", "direction", "exit_time", "net_return"]
        )
    )
    trades.to_csv(output_dir / "trades.csv", index=False)
    summaries = []
    for strategy in ["BASE", "QUALITY"]:
        selected = trades.loc[trades.strategy == strategy].sort_values("exit_time")
        for side in ["ALL", "BUY", "SELL"]:
            subset = (
                selected if side == "ALL" else selected.loc[selected.direction == side]
            )
            summaries.append(
                {
                    "strategy": strategy,
                    "side": side,
                    **strategy_stats(subset.net_return),
                }
            )
    summary = pd.DataFrame(summaries)
    summary.to_csv(output_dir / "summary.csv", index=False)
    if importance_frames:
        pd.concat(importance_frames).to_csv(
            output_dir / "quality_importance.csv", index=False
        )
        pd.concat(score_frames).to_csv(output_dir / "test_scores.csv", index=False)
    metadata["status"] = "completed" if trade_frames else "no_evaluable_folds"
    metadata["evaluated_folds"] = sum(r["status"] == "evaluated" for r in rows)
    save_metadata()
    return summary





# ====================================================================
# 13. データ取得（CSV未指定のときだけネットワークを使う）
# ====================================================================
import contextlib
import sys
import traceback
import uuid
import zipfile
from datetime import datetime, timezone


def download_snapshot(path):
    """直近59暦日の5分足を取得し、未確定足とOHLC欠損行を除いて固定保存。"""
    import yfinance as yf

    now = pd.Timestamp.now(tz="UTC")
    start = now - pd.Timedelta(days=59)
    print("USD/JPY 5分足を取得しています...", flush=True)
    raw = yf.download(
        "JPY=X", start=start.to_pydatetime(), end=now.to_pydatetime(),
        interval="5m", auto_adjust=False, progress=False, threads=False,
        ignore_tz=False, keepna=True, multi_level_index=False, timeout=30,
    )
    if raw is None or raw.empty:
        raise RuntimeError("価格データを取得できませんでした。通信・取得制限を確認し、後で再試行するか --csv を指定してください。")
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    if raw.index.tz is None:
        raise ValueError("取得データのタイムゾーンが不明です。推測で補わず停止します。")
    raw = raw[["Open", "High", "Low", "Close"]]
    missing = int(raw.isna().any(axis=1).sum())
    raw = raw.dropna()
    incomplete = raw.index.tz_convert("UTC") + pd.Timedelta(minutes=5) > now
    incomplete_count = int(incomplete.sum())
    raw = raw.loc[~incomplete].copy()
    raw.index = raw.index.tz_convert("Asia/Tokyo")
    raw.to_csv(path, index_label="timestamp")
    load_bars(path)  # 重複、逆順、不正OHLCをそのまま通さない。
    return {
        "source": "Yahoo Finance via yfinance", "symbol": "JPY=X",
        "interval": "5m", "yfinance_version": importlib.metadata.version("yfinance"),
        "requested_start_utc": str(start), "retrieved_at_utc": str(now),
        "missing_ohlc_rows_removed": missing,
        "incomplete_bars_removed": incomplete_count,
        "note": "New snapshot; not a reproduction of the old notebook period.",
    }


# ====================================================================
# 14. スコア別・方向別・コスト別の診断（Testの設定変更には使わない）
# ====================================================================
def write_diagnostics(results):
    summary = pd.read_csv(results / "summary.csv")
    folds = pd.read_csv(results / "folds.csv")
    trades = pd.read_csv(results / "trades.csv")
    report = [
        "# BASE / QUALITY 検証結果", "",
        "今回の新しい固定データによる研究結果です。旧Notebookの成績の再現ではありません。",
        "CSVの損益・勝率は小数単位（0.001 = 0.1%）。下の表示だけ%へ変換しています。", "",
        "## 全体・売買方向別", "",
    ]
    shown = summary.copy()
    for col in ["win_rate", "avg_return", "max_dd", "total_growth"]:
        shown[col] = shown[col] * 100
    shown = shown.rename(columns={"win_rate": "win_rate_pct", "avg_return": "avg_return_pct",
                                   "max_dd": "max_dd_pct", "total_growth": "total_growth_pct"})
    shown.to_csv(results / "summary_percent.csv", index=False)
    report += ["```text", shown.to_string(index=False, float_format=lambda x: f"{x:.6f}"), "```", "",
               "## Foldの状態", "", "```text", folds.to_string(index=False), "```", ""]
    if trades.empty:
        report += ["評価可能な取引がありません。folds.csvのreasonを確認してください。",
                   "取引数を増やすために同じTestを見ながら閾値を調整しないでください。"]
        (results / "REPORT.md").write_text("\n".join(report), encoding="utf-8")
        print(shown.to_string(index=False))
        return

    # mergeはfoldと時刻の両方で対応づける。違うfoldのスコアを混ぜない。
    scores = pd.read_csv(results / "test_scores.csv")
    trades["signal_time"] = pd.to_datetime(trades.signal_time, utc=True)
    scores["signal_time"] = pd.to_datetime(scores.signal_time, utc=True)
    scored = trades.merge(scores, on=["fold", "signal_time"], how="left", validate="many_to_one")
    if scored.p_quality.isna().any():
        raise ValueError("取引とQualityスコアの対応に欠損があります。")
    scored.to_csv(results / "trades_with_scores.csv", index=False)

    # BASEで実行された取引を固定母集団にする。全候補や独立サンプルの分析ではない。
    base = scored.loc[scored.strategy == "BASE"].copy()
    edges = [0, .40, .50, .55, .60, .65, .70, np.nextafter(1.0, 2.0)]
    labels = ["[0,.40)", "[.40,.50)", "[.50,.55)", "[.55,.60)",
              "[.60,.65)", "[.65,.70)", "[.70,1]"]
    base["score_band"] = pd.cut(base.p_quality, edges, labels=labels, right=False)
    band_rows = []
    for fold_key in ["ALL"] + sorted(base.fold.unique().tolist()):
        for side in ["ALL", "BUY", "SELL"]:
            selected = base if fold_key == "ALL" else base.loc[base.fold == fold_key]
            if side != "ALL":
                selected = selected.loc[selected.direction == side]
            for band in labels:
                group = selected.loc[selected.score_band == band].sort_values("exit_time")
                band_rows.append({"fold": fold_key, "side": side, "score_band": band,
                                  "mean_quality_score": group.p_quality.mean(),
                                  "small_sample_under_30": len(group) < 30,
                                  **strategy_stats(group.net_return)})
    bands = pd.DataFrame(band_rows)
    bands.to_csv(results / "quality_score_bands.csv", index=False)

    # 売買回数は固定したまま、往復コストだけ増加させる感度分析。
    # コストごとの再学習や閾値の再選択を行う別戦略の比較ではない。
    stress_rows = []
    for strategy in ["BASE", "QUALITY"]:
        selected = trades.loc[trades.strategy == strategy].sort_values("exit_time")
        for multiplier in [1.0, 1.5, 2.0]:
            stressed = selected.gross_return - selected.cost * multiplier
            stress_rows.append({"strategy": strategy, "cost_multiplier": multiplier,
                                **strategy_stats(stressed)})
    stress = pd.DataFrame(stress_rows)
    stress.to_csv(results / "cost_sensitivity.csv", index=False)

    # Equityは取引番号ではなく実際の決済時刻で並べる。
    # 取引がない時間は横ばい。含み損を反映するmark-to-marketではない。
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    equity_series = []
    first = pd.to_datetime(trades.entry_time, utc=True).min() - pd.Timedelta(seconds=1)
    last = pd.to_datetime(trades.exit_time, utc=True).max()
    for strategy in ["BASE", "QUALITY"]:
        selected = trades.loc[trades.strategy == strategy].copy()
        selected["exit_time"] = pd.to_datetime(selected.exit_time, utc=True)
        selected = selected.sort_values("exit_time")
        growth = pd.Series((1 + selected.net_return).cumprod().to_numpy(),
                           index=selected.exit_time, name=strategy)
        growth = growth.groupby(level=0).last()
        growth.loc[first] = 1.0
        if last not in growth.index:
            growth.loc[last] = growth.sort_index().iloc[-1]
        equity_series.append(growth.sort_index())
    equity = pd.concat(equity_series, axis=1).sort_index().ffill()
    equity.to_csv(results / "equity.csv", index_label="exit_time_utc")
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for strategy in ["BASE", "QUALITY"]:
        ax.step(equity.index, equity[strategy], where="post", label=strategy)
    ax.set(title="Out-of-sample closed-trade equity", ylabel="Growth of 1 (unlevered)", xlabel="Exit time (UTC)")
    ax.legend(); ax.grid(alpha=.25); fig.autofmt_xdate(); fig.tight_layout()
    fig.savefig(results / "equity.png", dpi=160); plt.close(fig)

    pooled = bands.loc[(bands.fold == "ALL") & (bands.side == "ALL")].copy()
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.bar(pooled.score_band, pooled.avg_return * 100)
    ax.axhline(0, color="black", linewidth=.8)
    ax.set(title="Quality score vs return: executed BASE trades only",
           xlabel="Fixed score bands (not calibration bins)", ylabel="Mean net return / trade (%)")
    for i, row in enumerate(pooled.itertuples()):
        ax.text(i, 0, f"n={row.trades}", ha="center", va="bottom", fontsize=8)
    ax.tick_params(axis="x", rotation=20); fig.tight_layout()
    fig.savefig(results / "quality_score_bands.png", dpi=160); plt.close(fig)

    totals = summary.loc[summary.side == "ALL"].set_index("strategy")
    b, q = totals.loc["BASE"], totals.loc["QUALITY"]
    evaluated = folds.loc[folds.status == "evaluated"]
    improved = int((evaluated.quality_avg_return > evaluated.base_avg_return).sum())
    reduction = 100 * (1 - q.trades / b.trades) if b.trades else float("nan")
    conclusion = "集計の平均純損益は改善しています。複数期間・コスト耐性・標本数の確認が必要です。" if q.avg_return > b.avg_return else "集計の平均純損益は改善していません。Qualityを有効とは判断できません。"
    report += ["## 確認事項", "", conclusion,
               f"- 平均純損益が改善したfold: {improved} / {len(evaluated)}（取引なしのfoldは改善に数えません）",
               f"- QUALITYのPF: {q.profit_factor:.6f}",
               f"- 取引削減率: {reduction:.2f}%",
               "- スコア別集計はBASEの実行取引が母集団です。BASEが見送った候補を含みません。",
               "- スコア帯は事前固定。空の帯・少数取引の帯も残しています。閾値の最適化結果ではありません。",
               "- Qualityスコアは校正済みの実勝率ではありません。",
               "- コスト感度は同じ取引に対する再計算で、別コストで再学習した成績ではありません。",
               "- DDは決済ベースです。リスク量の異なる取引を補正した比較ではありません。",
               "- 学習ラベルは時間決済の勝敗、評価損益はTP/SL込みであり、目的のずれが残っています。",
               "- 直近データは以前の検証期間と重なる可能性があります。独立した最終holdoutとは断定できません。",
               "", "## コスト感度（小数単位）", "", "```text", stress.to_string(index=False), "```",
               "", "## スコア別（BASE実行取引、小数単位）", "", "```text", pooled.to_string(index=False), "```",
               "", "![Equity](equity.png)", "", "![Score bands](quality_score_bands.png)", ""]
    (results / "REPORT.md").write_text("\n".join(report), encoding="utf-8")
    print("\n=== 最終結果（%表示） ===")
    print(shown.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
    print(conclusion)


# ====================================================================
# 15. 実行・ログ・返送用ZIP
# ====================================================================
class Tee:
    """画面の進捗を残しつつ、同じ内容をログにも書きます。"""
    def __init__(self, screen, log):
        self.screen, self.log = screen, log
    def write(self, text):
        self.screen.write(text)
        self.log.write(text)
        self.log.flush()
        return len(text)
    def flush(self):
        self.screen.flush()
        self.log.flush()


def run(csv_path=None, output_root=None):
    """Notebookからimportして run() を呼ぶこともできます。"""
    root = Path(output_root) if output_root else Path(__file__).resolve().parent / "fx_experiment_runs"
    run_dir = root / (datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6])
    run_dir.mkdir(parents=True, exist_ok=False)
    results = run_dir / "results"
    input_path = run_dir / "usdjpy_5m.csv"
    status = {"status": "running", "started_at_utc": datetime.now(timezone.utc).isoformat()}
    error = None
    with (run_dir / "console.log").open("w", encoding="utf-8") as log:
        with contextlib.redirect_stdout(Tee(sys.stdout, log)):
            try:
                if csv_path is None:
                    status["input"] = download_snapshot(input_path)
                else:
                    bars = load_bars(Path(csv_path))
                    bars.to_csv(input_path, index_label="timestamp")
                    status["input"] = {
                        "source": "user-supplied CSV (normalized snapshot)",
                        "original_filename": Path(csv_path).name,
                        "original_sha256": hashlib.sha256(Path(csv_path).read_bytes()).hexdigest(),
                    }
                print(f"固定保存したデータ: {input_path.name}")
                print("5 foldの学習を開始します。Validationの探索中はしばらく表示が止まることがあります。", flush=True)
                run_experiment(input_path, results)
                write_diagnostics(results)
                status["status"] = json.loads((results / "run.json").read_text())["status"]
            except Exception as exc:
                error = exc
                status["status"] = "failed"
                status["error_type"] = type(exc).__name__
                safe_trace = traceback.format_exc().replace(str(Path.home()), "[USER_HOME]")
                (run_dir / "error.txt").write_text(safe_trace, encoding="utf-8")
                print(f"停止しました: {type(exc).__name__}。error.txtを結果ZIPに保存します。")
    status["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
    (run_dir / "status.json").write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding="utf-8")
    zip_path = run_dir / "SEND_ME_results.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(results.rglob("*")) if results.exists() else []:
            if path.is_file():
                archive.write(path, path.relative_to(run_dir))
        for filename in ["console.log", "status.json", "error.txt"]:
            if (run_dir / filename).exists():
                archive.write(run_dir / filename, filename)
    print(f"\n送ってほしいファイル: {zip_path.resolve()}")
    if input_path.exists():
        print(f"価格CSVは手元に保存されています: {input_path.resolve()}")
    if error is not None:
        print("検証は未完了です。結果ZIPを送ってください。原因を確認します。")
    return zip_path


def main():
    parser = argparse.ArgumentParser(description="USD/JPY BASE / QUALITY検証と結果ZIP作成")
    parser.add_argument("--csv", type=Path, help="固定OHLC CSV。省略時は直近59日を取得します。")
    parser.add_argument("--out-root", type=Path, help="実行結果フォルダの親ディレクトリ")
    args = parser.parse_args()
    run(args.csv, args.out_root)


if __name__ == "__main__":
    main()
    


## 元のセル index 17


In [ ]:
%run fx_next_experiment.py

## 元のセル index 18


In [ ]:
# -*- coding: utf-8 -*-
"""USD/JPY: 次に実行するBASE対QUALITY検証（1ファイル版）。

最初に一度インストール:
    python -m pip install numpy pandas scikit-learn matplotlib yfinance

実行:
    python fx_next_experiment.py

Jupyterではこのコード全体を1セルに貼って実行してOK。

【何を確認するか】
1. BASE / QUALITYの未見期間の平均純損益・PF・DD・取引数。
2. Qualityスコアが高いBASE取引ほど実損益が良いか。
3. BUY / SELL、foldごとの違い。
4. コストを1.5倍・2倍にしたときの損益感度。

【ルール】
シグナル足確定 → 次足Open → 最大6本目Close。
同一足TP/SLはSL先行。
SELLはエントリー元本基準。
Qualityの正解は「方向付き時間決済リターン－コスト > 0」。
Testは設定選択に使わない。
"""

from __future__ import annotations

# ============================================================
# 1. 固定設定
# ============================================================

HORIZON_BARS = 6
MOVE_THRESHOLD = 0.0005
TRADING_COST = 1.33e-05
HALF_LIFE_DAYS = 20

OUTER_SPLITS = 5
QUALITY_OOF_SPLITS = 4

MIN_VALIDATION_TRADES = 15

MOVE_PROB_LIST = [
    0.55,
    0.60,
    0.65,
    0.70,
]

DIRECTION_PROB_LIST = [
    0.55,
    0.60,
    0.65,
    0.70,
]

QUALITY_PROB_LIST = [
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
]

TP_LIST = [
    0.0005,
    0.0008,
    0.0010,
]

SL_LIST = [
    0.0005,
    0.0007,
    0.0010,
]


# ============================================================
# 2. 特徴量一覧
# ============================================================

move_features = [
    "volatility_1h",
    "volatility_2h",
    "volatility_4h",
    "range",
    "body",
    "return_5m",
    "return_15m",
    "return_30m",
    "MA20_slope",
    "MA50_slope",
    "distance_high_1h",
    "distance_low_1h",
    "hour_sin",
    "hour_cos",
    "weekday",
]

direction_features = [
    "return_5m",
    "return_15m",
    "return_30m",
    "return_1h",
    "return_2h",
    "MA5_distance",
    "MA20_distance",
    "MA50_distance",
    "MA5_slope",
    "MA20_slope",
    "MA50_slope",
    "RSI",
    "bullish",
    "body",
    "upper_wick",
    "lower_wick",
    "distance_high_1h",
    "distance_low_1h",
    "volatility_1h",
    "hour_sin",
    "hour_cos",
    "weekday",
]

quality_market_features = [
    "return_5m",
    "return_15m",
    "return_30m",
    "return_1h",
    "return_2h",
    "volatility_1h",
    "volatility_2h",
    "volatility_4h",
    "ATR14_pct",
    "ADX14",
    "RSI",
    "MA5_distance",
    "MA20_distance",
    "MA50_distance",
    "MA5_slope",
    "MA20_slope",
    "MA50_slope",
    "distance_high_1h",
    "distance_low_1h",
    "body",
    "range",
    "upper_wick",
    "lower_wick",
    "hour_sin",
    "hour_cos",
    "weekday",
]


# ============================================================
# 3. ライブラリ
# ============================================================

import argparse
import contextlib
import hashlib
import importlib.metadata
import itertools
import json
import platform
import sys
import traceback
import uuid
import zipfile

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier


# ============================================================
# 4. CSV読み込み
# ============================================================

def load_bars(path: str | Path) -> pd.DataFrame:
    """
    CSVを読み込み、時刻を日本時間へ統一する。
    """

    frame = pd.read_csv(path)

    required = [
        "timestamp",
        "Open",
        "High",
        "Low",
        "Close",
    ]

    if not set(required).issubset(frame.columns):
        raise ValueError(
            f"CSV must contain {required}"
        )

    timestamps = (
        frame.pop("timestamp")
        .astype(str)
    )

    if not timestamps.str.contains(
        r"(?:Z|[+-]\d{2}:?\d{2})$",
        regex=True,
    ).all():
        raise ValueError(
            "Every timestamp must include Z or a UTC offset."
        )

    frame.index = (
        pd.DatetimeIndex(
            pd.to_datetime(
                timestamps,
                utc=True,
            )
        )
        .tz_convert(
            "Asia/Tokyo"
        )
    )

    frame = (
        frame[
            [
                "Open",
                "High",
                "Low",
                "Close",
            ]
        ]
        .apply(
            pd.to_numeric,
            errors="raise",
        )
    )

    if (
        frame.empty
        or not frame.index.is_monotonic_increasing
        or not frame.index.is_unique
    ):
        raise ValueError(
            "Bars must be nonempty, chronological and unique."
        )

    if (
        not np.isfinite(
            frame.to_numpy()
        ).all()
        or
        (frame <= 0).any().any()
    ):
        raise ValueError(
            "OHLC must be finite and positive."
        )

    if (
        frame["High"]
        <
        frame[
            [
                "Open",
                "Close",
                "Low",
            ]
        ].max(axis=1)
    ).any():
        raise ValueError(
            "High is inconsistent with OHLC."
        )

    if (
        frame["Low"]
        >
        frame[
            [
                "Open",
                "Close",
                "High",
            ]
        ].min(axis=1)
    ).any():
        raise ValueError(
            "Low is inconsistent with OHLC."
        )

    invalid_grid = (
        (frame.index.minute % 5 != 0)
        |
        (frame.index.second != 0)
        |
        (frame.index.microsecond != 0)
        |
        (frame.index.nanosecond != 0)
    )

    if invalid_grid.any():
        raise ValueError(
            "Bar timestamps must align to a five-minute grid."
        )

    return frame


# ============================================================
# 5. 特徴量
# ============================================================

def make_features(bars):
    """
    シグナル時点までの過去データだけで特徴量を作る。
    """

    df = bars.copy()

    # ----------------------------
    # リターン
    # ----------------------------

    df["return_5m"] = (
        df["Close"]
        .pct_change(1)
    )

    df["return_15m"] = (
        df["Close"]
        .pct_change(3)
    )

    df["return_30m"] = (
        df["Close"]
        .pct_change(6)
    )

    df["return_1h"] = (
        df["Close"]
        .pct_change(12)
    )

    df["return_2h"] = (
        df["Close"]
        .pct_change(24)
    )

    # ----------------------------
    # 移動平均
    # ----------------------------

    df["MA5"] = (
        df["Close"]
        .rolling(5)
        .mean()
    )

    df["MA20"] = (
        df["Close"]
        .rolling(20)
        .mean()
    )

    df["MA50"] = (
        df["Close"]
        .rolling(50)
        .mean()
    )

    df["MA5_distance"] = (
        df["Close"]
        / df["MA5"]
        - 1
    )

    df["MA20_distance"] = (
        df["Close"]
        / df["MA20"]
        - 1
    )

    df["MA50_distance"] = (
        df["Close"]
        / df["MA50"]
        - 1
    )

    df["MA5_slope"] = (
        df["MA5"]
        .pct_change(3)
    )

    df["MA20_slope"] = (
        df["MA20"]
        .pct_change(3)
    )

    df["MA50_slope"] = (
        df["MA50"]
        .pct_change(3)
    )

    # ----------------------------
    # ローソク足
    # ----------------------------

    df["body"] = (
        abs(
            df["Close"]
            - df["Open"]
        )
        / df["Open"]
    )

    df["range"] = (
        (
            df["High"]
            - df["Low"]
        )
        / df["Close"]
    )

    df["upper_wick"] = (
        (
            df["High"]
            -
            df[
                [
                    "Open",
                    "Close",
                ]
            ].max(axis=1)
        )
        / df["Close"]
    )

    df["lower_wick"] = (
        (
            df[
                [
                    "Open",
                    "Close",
                ]
            ].min(axis=1)
            - df["Low"]
        )
        / df["Close"]
    )

    df["bullish"] = (
        df["Close"]
        > df["Open"]
    ).astype(int)

    # ----------------------------
    # ボラティリティ
    # ----------------------------

    df["volatility_1h"] = (
        df["return_5m"]
        .rolling(12)
        .std()
    )

    df["volatility_2h"] = (
        df["return_5m"]
        .rolling(24)
        .std()
    )

    df["volatility_4h"] = (
        df["return_5m"]
        .rolling(48)
        .std()
    )

    # ----------------------------
    # RSI
    # ----------------------------

    delta = (
        df["Close"]
        .diff()
    )

    gain = (
        delta.clip(
            lower=0
        )
    )

    loss = (
        -delta.clip(
            upper=0
        )
    )

    avg_gain = (
        gain
        .rolling(14)
        .mean()
    )

    avg_loss = (
        loss
        .rolling(14)
        .mean()
    )

    rs = (
        avg_gain
        / avg_loss
    )

    df["RSI"] = (
        100
        -
        100
        / (
            1 + rs
        )
    )

    # ----------------------------
    # 高値・安値との距離
    # ----------------------------

    df["high_1h"] = (
        df["High"]
        .rolling(12)
        .max()
    )

    df["low_1h"] = (
        df["Low"]
        .rolling(12)
        .min()
    )

    df["distance_high_1h"] = (
        df["Close"]
        / df["high_1h"]
        - 1
    )

    df["distance_low_1h"] = (
        df["Close"]
        / df["low_1h"]
        - 1
    )

    # ----------------------------
    # 時間
    # ----------------------------

    df["hour"] = (
        df.index.hour
    )

    df["weekday"] = (
        df.index.dayofweek
    )

    df["hour_sin"] = np.sin(
        2
        * np.pi
        * df["hour"]
        / 24
    )

    df["hour_cos"] = np.cos(
        2
        * np.pi
        * df["hour"]
        / 24
    )

    # ----------------------------
    # ATR
    # ----------------------------

    previous_close = (
        df["Close"]
        .shift(1)
    )

    tr1 = (
        df["High"]
        - df["Low"]
    )

    tr2 = abs(
        df["High"]
        - previous_close
    )

    tr3 = abs(
        df["Low"]
        - previous_close
    )

    true_range = pd.concat(
        [
            tr1,
            tr2,
            tr3,
        ],
        axis=1,
    ).max(
        axis=1
    )

    df["ATR14"] = (
        true_range
        .rolling(14)
        .mean()
    )

    df["ATR14_pct"] = (
        df["ATR14"]
        / df["Close"]
    )

    # ----------------------------
    # ADX
    # ----------------------------

    high_diff = (
        df["High"]
        .diff()
    )

    low_diff = (
        -df["Low"]
        .diff()
    )

    plus_dm = np.where(
        (
            high_diff
            > low_diff
        )
        &
        (
            high_diff
            > 0
        ),
        high_diff,
        0.0,
    )

    minus_dm = np.where(
        (
            low_diff
            > high_diff
        )
        &
        (
            low_diff
            > 0
        ),
        low_diff,
        0.0,
    )

    plus_dm = pd.Series(
        plus_dm,
        index=df.index,
    )

    minus_dm = pd.Series(
        minus_dm,
        index=df.index,
    )

    atr_adx = (
        true_range
        .rolling(14)
        .mean()
    )

    plus_di = (
        100
        *
        plus_dm
        .rolling(14)
        .mean()
        /
        atr_adx
    )

    minus_di = (
        100
        *
        minus_dm
        .rolling(14)
        .mean()
        /
        atr_adx
    )

    dx = (
        100
        *
        abs(
            plus_di
            - minus_di
        )
        /
        (
            plus_di
            + minus_di
        )
    )

    df["ADX14"] = (
        dx
        .rolling(14)
        .mean()
    )

    return df.replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )


# ============================================================
# 6. 教師ラベル
# ============================================================

def prepare_data(bars):
    """
    tでシグナル。
    t+1 OpenでEntry。
    t+6 Closeで30分後Exit。
    """

    frame = (
        make_features(
            bars
        )
    )

    frame["bar_position"] = (
        np.arange(
            len(bars)
        )
    )

    frame["entry_price"] = (
        bars["Open"]
        .shift(-1)
    )

    frame["exit_price"] = (
        bars["Close"]
        .shift(
            -HORIZON_BARS
        )
    )

    frame["future_return"] = (
        frame["exit_price"]
        / frame["entry_price"]
        - 1
    )

    times = pd.Series(
        bars.index,
        index=bars.index,
    )

    frame["label_end"] = (
        times.shift(
            -HORIZON_BARS
        )
        +
        pd.Timedelta(
            minutes=5
        )
    )

    complete = (
        times.shift(
            -HORIZON_BARS
        )
        - times
    ).eq(
        pd.Timedelta(
            minutes=
                5
                * HORIZON_BARS
        )
    )

    frame["move_target"] = (
        frame["future_return"]
        .abs()
        .gt(
            MOVE_THRESHOLD
        )
        .astype(int)
    )

    frame["direction_target"] = (
        frame["future_return"]
        .gt(0)
        .astype(int)
    )

    required = list(
        dict.fromkeys(
            move_features
            +
            direction_features
            +
            quality_market_features
        )
    )

    required += [
        "future_return",
        "entry_price",
        "exit_price",
        "label_end",
    ]

    return (
        frame
        .loc[complete]
        .dropna(
            subset=required
        )
        .copy()
    )


# ============================================================
# 7. 時系列Fold
# ============================================================

@dataclass(
    frozen=True
)
class Fold:
    number: int
    train: pd.DataFrame
    core: pd.DataFrame
    validation: pd.DataFrame
    test: pd.DataFrame


def outer_folds(data):

    block = (
        len(data)
        //
        (
            OUTER_SPLITS
            + 1
        )
    )

    if block < 1:
        return

    for number in range(
        1,
        OUTER_SPLITS + 1,
    ):

        train_end = (
            block
            * number
        )

        test_start = (
            train_end
            + HORIZON_BARS
        )

        test_end = min(
            test_start
            + block,
            len(data),
        )

        train = (
            data.iloc[
                :train_end
            ]
        )

        cut = int(
            len(train)
            * 0.8
        )

        core = (
            train.iloc[
                :
                max(
                    0,
                    cut
                    - HORIZON_BARS
                )
            ]
        )

        validation = (
            train.iloc[
                cut:
            ]
        )

        test = (
            data.iloc[
                test_start:test_end
            ]
        )

        if (
            len(validation)
            and
            len(core)
        ):

            if not (
                core["label_end"]
                <=
                validation.index[0]
            ).all():

                raise ValueError(
                    "Training label crosses validation boundary."
                )

        if (
            len(test)
            and
            len(train)
        ):

            if not (
                train["label_end"]
                <=
                test.index[0]
            ).all():

                raise ValueError(
                    "Training label crosses test boundary."
                )

        yield Fold(
            number,
            train,
            core,
            validation,
            test,
        )


# ============================================================
# 8. 時間減衰
# ============================================================

def make_time_weights(
    index,
    half_life_days,
):

    latest = (
        index.max()
    )

    age_days = (
        (
            latest
            - index
        )
        .total_seconds()
        / 86400
    )

    weights = (
        0.5
        **
        (
            age_days
            / half_life_days
        )
    )

    return np.array(
        weights
    )


# ============================================================
# 9. MOVE + Directionモデル
# ============================================================

def fit_base_models(
    train_frame,
    trees=250,
):

    if (
        train_frame[
            "move_target"
        ].nunique()
        < 2
    ):
        return None

    move_weights = (
        make_time_weights(
            train_frame.index,
            HALF_LIFE_DAYS,
        )
    )

    move_model = (
        RandomForestClassifier(
            n_estimators=
                trees,
            max_depth=
                8,
            min_samples_leaf=
                20,
            max_features=
                "sqrt",
            class_weight=
                "balanced",
            random_state=
                42,
            n_jobs=
                -1,
        )
    )

    move_model.fit(
        train_frame[
            move_features
        ],
        train_frame[
            "move_target"
        ],
        sample_weight=
            move_weights,
    )

    direction_train = (
        train_frame[
            train_frame[
                "move_target"
            ]
            == 1
        ]
    )

    if (
        len(direction_train)
        < 50
        or
        direction_train[
            "direction_target"
        ].nunique()
        < 2
    ):
        return None

    direction_weights = (
        make_time_weights(
            direction_train.index,
            HALF_LIFE_DAYS,
        )
    )

    direction_model = (
        RandomForestClassifier(
            n_estimators=
                trees,
            max_depth=
                8,
            min_samples_leaf=
                15,
            max_features=
                "sqrt",
            class_weight=
                "balanced",
            random_state=
                42,
            n_jobs=
                -1,
        )
    )

    direction_model.fit(
        direction_train[
            direction_features
        ],
        direction_train[
            "direction_target"
        ],
        sample_weight=
            direction_weights,
    )

    return (
        move_model,
        direction_model,
    )


def predict_base_models(
    models,
    frame,
):

    move_model, \
    direction_model = models

    p_move = (
        move_model
        .predict_proba(
            frame[
                move_features
            ]
        )[:, 1]
    )

    direction_prob = (
        direction_model
        .predict_proba(
            frame[
                direction_features
            ]
        )
    )

    class_map = {
        c: i
        for i, c
        in enumerate(
            direction_model.classes_
        )
    }

    p_down = (
        direction_prob[
            :,
            class_map[0],
        ]
    )

    p_up = (
        direction_prob[
            :,
            class_map[1],
        ]
    )

    return (
        p_move,
        p_up,
        p_down,
    )


# ============================================================
# 10. Quality特徴量
# ============================================================

def build_quality_features(
    frame,
    p_move,
    p_up,
    p_down,
):

    quality_x = (
        frame[
            quality_market_features
        ]
        .copy()
    )

    quality_x[
        "p_move"
    ] = p_move

    quality_x[
        "p_up"
    ] = p_up

    quality_x[
        "p_down"
    ] = p_down

    quality_x[
        "direction_confidence"
    ] = abs(
        p_up
        - p_down
    )

    quality_x[
        "predicted_direction"
    ] = (
        p_up
        >= p_down
    ).astype(int)

    return quality_x


def make_quality_target(
    frame,
    p_up,
    p_down,
):

    predicted_buy = (
        p_up
        >= p_down
    )

    directional_return = np.where(
        predicted_buy,
        frame[
            "future_return"
        ].values,
        -frame[
            "future_return"
        ].values,
    )

    net_return = (
        directional_return
        - TRADING_COST
    )

    quality_target = (
        net_return
        > 0
    ).astype(int)

    return (
        quality_target,
        net_return,
    )


# ============================================================
# 11. Quality用OOFデータ
# ============================================================

def create_quality_oof_dataset(
    frame
):

    initial_size = int(
        len(frame)
        * 0.4
    )

    remaining = (
        len(frame)
        - initial_size
    )

    block = max(
        remaining
        //
        QUALITY_OOF_SPLITS,
        1,
    )

    qx_list = []

    for split in range(
        QUALITY_OOF_SPLITS
    ):

        val_start = (
            initial_size
            +
            split
            * block
        )

        if (
            split
            ==
            QUALITY_OOF_SPLITS
            - 1
        ):
            val_end = (
                len(frame)
            )

        else:
            val_end = min(
                val_start
                + block,
                len(frame),
            )

        train_end = (
            val_start
            - HORIZON_BARS
        )

        if (
            train_end
            < 200
        ):
            continue

        oof_train = (
            frame.iloc[
                :train_end
            ]
        )

        oof_val = (
            frame.iloc[
                val_start:val_end
            ]
        )

        if len(
            oof_val
        ) == 0:
            continue

        models = (
            fit_base_models(
                oof_train,
                trees=180,
            )
        )

        if models is None:
            continue

        (
            p_move,
            p_up,
            p_down,
        ) = (
            predict_base_models(
                models,
                oof_val,
            )
        )

        quality_x = (
            build_quality_features(
                oof_val,
                p_move,
                p_up,
                p_down,
            )
        )

        (
            quality_y,
            _,
        ) = (
            make_quality_target(
                oof_val,
                p_up,
                p_down,
            )
        )

        quality_x[
            "quality_target"
        ] = quality_y

        qx_list.append(
            quality_x
        )

    if (
        len(qx_list)
        == 0
    ):
        return None

    quality_data = (
        pd.concat(
            qx_list
        )
        .sort_index()
    )

    return quality_data


# ============================================================
# 12. Qualityモデル
# ============================================================

def fit_quality_model(
    quality_data
):

    quality_feature_names = [
        c
        for c
        in quality_data.columns
        if c
        != "quality_target"
    ]

    if (
        len(quality_data)
        < 100
        or
        quality_data[
            "quality_target"
        ].nunique()
        < 2
    ):
        return None

    weights = (
        make_time_weights(
            quality_data.index,
            HALF_LIFE_DAYS,
        )
    )

    model = (
        RandomForestClassifier(
            n_estimators=
                300,
            max_depth=
                7,
            min_samples_leaf=
                20,
            max_features=
                "sqrt",
            class_weight=
                "balanced",
            random_state=
                123,
            n_jobs=
                -1,
        )
    )

    model.fit(
        quality_data[
            quality_feature_names
        ],
        quality_data[
            "quality_target"
        ],
        sample_weight=
            weights,
    )

    return (
        model,
        quality_feature_names,
    )


def predict_quality(
    quality_bundle,
    frame,
    p_move,
    p_up,
    p_down,
):

    model, \
    feature_names = (
        quality_bundle
    )

    x = (
        build_quality_features(
            frame,
            p_move,
            p_up,
            p_down,
        )
    )

    probability = (
        model
        .predict_proba(
            x[
                feature_names
            ]
        )[:, 1]
    )

    return probability


# ============================================================
# 13. BUY / SELL / WAIT
# ============================================================

def make_signals(
    p_move,
    p_up,
    p_down,
    move_t,
    direction_t,
    quality_prob=None,
    quality_t=None,
):

    buy = (
        (p_move >= move_t)
        &
        (p_up >= direction_t)
        &
        (p_up > p_down)
    )

    sell = (
        (p_move >= move_t)
        &
        (p_down >= direction_t)
        &
        (p_down > p_up)
    )

    if (
        quality_prob
        is not None
        and
        quality_t
        is not None
    ):

        quality_mask = (
            quality_prob
            >= quality_t
        )

        buy = (
            buy
            & quality_mask
        )

        sell = (
            sell
            & quality_mask
        )

    signals = np.zeros(
        len(p_move)
    )

    signals[
        buy
    ] = 1

    signals[
        sell
    ] = -1

    return signals


# ============================================================
# 14. 取引シミュレーション
# ============================================================

def simulate_trade(
    bars,
    signal_time,
    direction,
    tp,
    sl,
    *,
    end_time=None,
    cost=TRADING_COST,
):

    if (
        direction
        not in (
            "BUY",
            "SELL",
        )
    ):
        raise ValueError(
            "direction must be BUY or SELL"
        )

    if not (
        np.isfinite(
            [
                tp,
                sl,
                cost,
            ]
        ).all()
        and
        0 < tp < 1
        and
        0 < sl < 1
        and
        cost >= 0
    ):
        raise ValueError(
            "Require 0 < TP/SL < 1 and cost >= 0."
        )

    signal_loc = (
        bars.index
        .get_loc(
            signal_time
        )
    )

    final_loc = (
        signal_loc
        + HORIZON_BARS
    )

    if (
        final_loc
        >= len(bars)
    ):
        return None

    final_time = (
        bars.index[
            final_loc
        ]
        +
        pd.Timedelta(
            minutes=5
        )
    )

    if (
        end_time
        is not None
        and
        final_time
        > end_time
    ):
        return None

    if (
        bars.index[
            final_loc
        ]
        -
        bars.index[
            signal_loc
        ]
        !=
        pd.Timedelta(
            minutes=
                5
                * HORIZON_BARS
        )
    ):
        return None

    side = (
        1
        if direction
        == "BUY"
        else -1
    )

    entry = float(
        bars.iloc[
            signal_loc + 1
        ]["Open"]
    )

    exit_loc = (
        final_loc
    )

    exit_price = float(
        bars.iloc[
            final_loc
        ]["Close"]
    )

    reason = "TIME"

    for loc in range(
        signal_loc + 1,
        final_loc + 1,
    ):

        bar = (
            bars.iloc[
                loc
            ]
        )

        open_return = (
            side
            *
            (
                float(
                    bar["Open"]
                )
                / entry
                - 1
            )
        )

        # ------------------------
        # ギャップでSL超過
        # ------------------------

        if (
            open_return
            <= -sl
        ):

            exit_loc = loc

            exit_price = float(
                bar["Open"]
            )

            reason = "GAP_SL"

            break

        # ------------------------
        # Open時点でTP超過
        # ------------------------

        if (
            open_return
            >= tp
        ):

            exit_loc = loc

            exit_price = (
                entry
                *
                (
                    1
                    + side
                    * tp
                )
            )

            reason = "TP"

            break

        favorable = (
            side
            *
            (
                float(
                    bar[
                        "High"
                        if side == 1
                        else "Low"
                    ]
                )
                / entry
                - 1
            )
        )

        adverse = (
            side
            *
            (
                float(
                    bar[
                        "Low"
                        if side == 1
                        else "High"
                    ]
                )
                / entry
                - 1
            )
        )

        # 同一足でTP/SL両方なら
        # 保守的にSLを先に判定
        if (
            adverse
            <= -sl
        ):

            exit_loc = loc

            exit_price = (
                entry
                *
                (
                    1
                    - side
                    * sl
                )
            )

            reason = "SL"

            break

        if (
            favorable
            >= tp
        ):

            exit_loc = loc

            exit_price = (
                entry
                *
                (
                    1
                    + side
                    * tp
                )
            )

            reason = "TP"

            break

    gross = (
        side
        *
        (
            exit_price
            / entry
            - 1
        )
    )

    exit_time = (
        bars.index[
            exit_loc
        ]
        +
        pd.Timedelta(
            minutes=5
        )
    )

    return {
        "signal_time":
            signal_time,

        "entry_time":
            bars.index[
                signal_loc + 1
            ],

        "exit_time":
            exit_time,

        "direction":
            direction,

        "entry_price":
            entry,

        "exit_price":
            exit_price,

        "exit_reason":
            reason,

        "gross_return":
            gross,

        "cost":
            cost,

        "net_return":
            gross - cost,

        "signal_position":
            signal_loc,
    }


# ============================================================
# 15. バックテスト
# ============================================================

def run_backtest(
    bars,
    frame,
    signals,
    tp,
    sl,
    *,
    end_time=None,
    cost=TRADING_COST,
):

    if (
        len(frame)
        != len(signals)
        or
        not np.isin(
            signals,
            [
                -1,
                0,
                1,
            ]
        ).all()
    ):
        raise ValueError(
            "Signals must align with frame."
        )

    records = []

    next_signal_position = -1

    for (
        time,
        signal,
    ) in zip(
        frame.index,
        signals,
    ):

        position = (
            bars.index
            .get_loc(
                time
            )
        )

        if (
            signal == 0
            or
            position
            < next_signal_position
        ):
            continue

        trade = (
            simulate_trade(
                bars,
                time,
                (
                    "BUY"
                    if signal == 1
                    else "SELL"
                ),
                tp,
                sl,
                end_time=
                    end_time,
                cost=
                    cost,
            )
        )

        if (
            trade
            is not None
        ):

            records.append(
                trade
            )

            next_signal_position = (
                position
                + HORIZON_BARS
            )

    columns = [
        "signal_time",
        "entry_time",
        "exit_time",
        "direction",
        "entry_price",
        "exit_price",
        "exit_reason",
        "gross_return",
        "cost",
        "net_return",
        "signal_position",
    ]

    return (
        pd.DataFrame
        .from_records(
            records,
            columns=columns,
        )
    )


# ============================================================
# 16. 統計
# ============================================================

def strategy_stats(
    returns
):

    r = np.asarray(
        returns,
        dtype=float,
    )

    if (
        not np.isfinite(
            r
        ).all()
        or
        (r <= -1).any()
    ):
        raise ValueError(
            "Returns must be finite."
        )

    if not len(r):

        return dict(
            trades=0,
            win_rate=np.nan,
            avg_return=np.nan,
            profit_factor=np.nan,
            max_dd=np.nan,
            total_growth=0.0,
        )

    gains = (
        r[
            r > 0
        ].sum()
    )

    losses = (
        -r[
            r < 0
        ].sum()
    )

    if losses:

        pf = (
            gains
            / losses
        )

    elif gains:

        pf = np.inf

    else:

        pf = np.nan

    equity = np.r_[
        1.0,
        np.cumprod(
            1 + r
        ),
    ]

    drawdown = (
        equity
        /
        np.maximum.accumulate(
            equity
        )
        - 1
    )

    return dict(
        trades=
            len(r),

        win_rate=
            float(
                (
                    r > 0
                ).mean()
            ),

        avg_return=
            float(
                r.mean()
            ),

        profit_factor=
            float(
                pf
            ),

        max_dd=
            float(
                drawdown.min()
            ),

        total_growth=
            float(
                equity[-1]
                - 1
            ),
    )


# ============================================================
# 17. 実験本体
# ============================================================

def run_experiment(
    csv_path,
    output_dir,
):

    csv_path = Path(
        csv_path
    )

    output_dir = Path(
        output_dir
    )

    bars = (
        load_bars(
            csv_path
        )
    )

    data = (
        prepare_data(
            bars
        )
    )

    print(
        "取得した5分足:",
        len(bars)
    )

    print(
        "機械学習使用可能:",
        len(data)
    )

    if (
        len(data)
        < 600
    ):
        raise ValueError(
            "Need at least 600 usable rows."
        )

    output_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    versions = {
        name:
            importlib.metadata.version(
                name
            )
        for name
        in [
            "numpy",
            "pandas",
            "scikit-learn",
        ]
    }

    source_path = Path(
        __file__
    ) if "__file__" in globals() else None

    if (
        source_path
        is not None
        and
        source_path.exists()
    ):

        source_hashes = {
            source_path.name:
                hashlib.sha256(
                    source_path.read_bytes()
                ).hexdigest()
        }

    else:

        source_hashes = {
            "source":
                "Jupyter cell"
        }

    metadata = {
        "status":
            "running",

        "experiment":
            "quality-v0.2-jupyter-safe",

        "input_filename":
            csv_path.name,

        "input_sha256":
            hashlib.sha256(
                csv_path.read_bytes()
            ).hexdigest(),

        "bar_rows":
            len(bars),

        "usable_rows":
            len(data),

        "first_bar":
            str(
                bars.index[0]
            ),

        "last_bar":
            str(
                bars.index[-1]
            ),

        "timezone":
            "Asia/Tokyo",

        "python":
            platform.python_version(),

        "versions":
            versions,

        "source_sha256":
            source_hashes,

        "config":
            {
                "HORIZON_BARS":
                    HORIZON_BARS,

                "MOVE_THRESHOLD":
                    MOVE_THRESHOLD,

                "TRADING_COST":
                    TRADING_COST,

                "HALF_LIFE_DAYS":
                    HALF_LIFE_DAYS,

                "OUTER_SPLITS":
                    OUTER_SPLITS,

                "QUALITY_OOF_SPLITS":
                    QUALITY_OOF_SPLITS,

                "MIN_VALIDATION_TRADES":
                    MIN_VALIDATION_TRADES,

                "MOVE_PROB_LIST":
                    MOVE_PROB_LIST,

                "DIRECTION_PROB_LIST":
                    DIRECTION_PROB_LIST,

                "QUALITY_PROB_LIST":
                    QUALITY_PROB_LIST,

                "TP_LIST":
                    TP_LIST,

                "SL_LIST":
                    SL_LIST,
            },
    }

    def save_metadata():

        (
            output_dir
            / "run.json"
        ).write_text(
            json.dumps(
                metadata,
                indent=2,
            ),
            encoding="utf-8",
        )

    save_metadata()

    rows = []

    trade_frames = []

    importance_frames = []

    score_frames = []

    for fold in outer_folds(
        data
    ):

        print(
            "\n=============================="
        )

        print(
            f"Fold {fold.number}"
        )

        print(
            "=============================="
        )

        row = {
            "fold":
                fold.number,

            "status":
                "skipped",

            "reason":
                "",

            "train_rows":
                len(
                    fold.train
                ),

            "test_rows":
                len(
                    fold.test
                ),

            "test_start":
                (
                    str(
                        fold.test.index[0]
                    )
                    if len(
                        fold.test
                    )
                    else ""
                ),

            "test_end":
                (
                    str(
                        fold.test.index[-1]
                    )
                    if len(
                        fold.test
                    )
                    else ""
                ),
        }

        rows.append(
            row
        )

        if (
            len(
                fold.train
            )
            < 500
            or
            not len(
                fold.test
            )
            or
            not len(
                fold.validation
            )
        ):

            row[
                "reason"
            ] = (
                "insufficient_rows"
            )

            continue

        print(
            "内部OOFとQuality学習中..."
        )

        oof = (
            create_quality_oof_dataset(
                fold.core
            )
        )

        quality = (
            fit_quality_model(
                oof
            )
            if oof
            is not None
            else None
        )

        base = (
            fit_base_models(
                fold.core,
                trees=300,
            )
        )

        if (
            quality
            is None
            or
            base
            is None
        ):

            row[
                "reason"
            ] = (
                "training_unavailable_or_one_class"
            )

            continue

        probabilities = (
            predict_base_models(
                base,
                fold.validation,
            )
        )

        quality_probabilities = (
            predict_quality(
                quality,
                fold.validation,
                *probabilities,
            )
        )

        val_end = (
            fold.validation.index[-1]
            +
            pd.Timedelta(
                minutes=5
            )
        )

        print(
            "Validationで閾値・TP/SLを選択中..."
        )

        best = None

        best_score = (
            -np.inf
        )

        for (
            move,
            direction,
            tp,
            sl,
        ) in itertools.product(
            MOVE_PROB_LIST,
            DIRECTION_PROB_LIST,
            TP_LIST,
            SL_LIST,
        ):

            signals = (
                make_signals(
                    *probabilities,
                    move,
                    direction,
                )
            )

            trades = (
                run_backtest(
                    bars,
                    fold.validation,
                    signals,
                    tp,
                    sl,
                    end_time=
                        val_end,
                )
            )

            if (
                len(trades)
                < MIN_VALIDATION_TRADES
            ):
                continue

            score = (
                trades[
                    "net_return"
                ].mean()
                *
                np.sqrt(
                    len(trades)
                )
            )

            if (
                score
                > best_score
            ):

                best_score = (
                    score
                )

                best = (
                    move,
                    direction,
                    tp,
                    sl,
                )

        if (
            best
            is None
        ):

            row[
                "reason"
            ] = (
                "no_base_setting_meets_min_validation_trades"
            )

            print(
                "Base設定を選べません"
            )

            continue

        (
            move,
            direction,
            tp,
            sl,
        ) = best

        print(
            "Base設定:",
            {
                "move_threshold":
                    move,

                "direction_threshold":
                    direction,

                "tp":
                    tp,

                "sl":
                    sl,
            }
        )

        best_quality = None

        quality_score = (
            -np.inf
        )

        for threshold in (
            QUALITY_PROB_LIST
        ):

            signals = (
                make_signals(
                    *probabilities,
                    move,
                    direction,
                    quality_prob=
                        quality_probabilities,
                    quality_t=
                        threshold,
                )
            )

            trades = (
                run_backtest(
                    bars,
                    fold.validation,
                    signals,
                    tp,
                    sl,
                    end_time=
                        val_end,
                )
            )

            if (
                len(trades)
                < MIN_VALIDATION_TRADES
            ):
                continue

            score = (
                trades[
                    "net_return"
                ].mean()
                *
                np.sqrt(
                    len(trades)
                )
            )

            if (
                score
                > quality_score
            ):

                quality_score = (
                    score
                )

                best_quality = (
                    threshold
                )

        row[
            "quality_filter_disabled"
        ] = (
            best_quality
            is None
        )

        threshold = (
            best_quality
            if best_quality
            is not None
            else 0.0
        )

        row.update(
            move_threshold=
                move,

            direction_threshold=
                direction,

            tp=
                tp,

            sl=
                sl,

            quality_threshold=
                threshold,
        )

        print(
            "Quality閾値:",
            threshold
        )

        print(
            "設定を固定して再学習 → Test評価中..."
        )

        final_oof = (
            create_quality_oof_dataset(
                fold.train
            )
        )

        final_quality = (
            fit_quality_model(
                final_oof
            )
            if final_oof
            is not None
            else None
        )

        final_base = (
            fit_base_models(
                fold.train,
                trees=400,
            )
        )

        if (
            final_quality
            is None
            or
            final_base
            is None
        ):

            row[
                "reason"
            ] = (
                "final_retraining_unavailable"
            )

            continue

        test_probabilities = (
            predict_base_models(
                final_base,
                fold.test,
            )
        )

        test_quality = (
            predict_quality(
                final_quality,
                fold.test,
                *test_probabilities,
            )
        )

        test_end = (
            fold.test.index[-1]
            +
            pd.Timedelta(
                minutes=5
            )
        )

        score_frames.append(
            pd.DataFrame(
                {
                    "fold":
                        fold.number,

                    "signal_time":
                        fold.test.index,

                    "p_move":
                        test_probabilities[
                            0
                        ],

                    "p_up":
                        test_probabilities[
                            1
                        ],

                    "p_down":
                        test_probabilities[
                            2
                        ],

                    "p_quality":
                        test_quality,
                }
            )
        )

        for (
            strategy,
            q,
        ) in [
            (
                "BASE",
                None,
            ),
            (
                "QUALITY",
                test_quality,
            ),
        ]:

            signals = (
                make_signals(
                    *test_probabilities,
                    move,
                    direction,
                    quality_prob=
                        q,
                    quality_t=
                        threshold,
                )
            )

            trades = (
                run_backtest(
                    bars,
                    fold.test,
                    signals,
                    tp,
                    sl,
                    end_time=
                        test_end,
                )
            )

            stats = (
                strategy_stats(
                    trades[
                        "net_return"
                    ]
                )
            )

            row.update(
                {
                    f"{strategy.lower()}_{k}":
                        v
                    for k, v
                    in stats.items()
                }
            )

            trades[
                "strategy"
            ] = strategy

            trades[
                "fold"
            ] = fold.number

            trade_frames.append(
                trades
            )

        model, names = (
            final_quality
        )

        importance_frames.append(
            pd.DataFrame(
                {
                    "fold":
                        fold.number,

                    "feature":
                        names,

                    "importance":
                        model.feature_importances_,
                }
            )
        )

        row[
            "status"
        ] = "evaluated"

        row[
            "reason"
        ] = ""

        print(
            f"Fold {fold.number}: "
            f"BASE {row['base_trades']} trades, "
            f"QUALITY {row['quality_trades']} trades"
        )

    folds_df = (
        pd.DataFrame(
            rows
        )
    )

    folds_df.to_csv(
        output_dir
        / "folds.csv",
        index=False,
    )

    if trade_frames:

        trades = (
            pd.concat(
                trade_frames,
                ignore_index=True,
            )
        )

    else:

        trades = (
            pd.DataFrame(
                columns=[
                    "strategy",
                    "fold",
                    "direction",
                    "exit_time",
                    "net_return",
                ]
            )
        )

    trades.to_csv(
        output_dir
        / "trades.csv",
        index=False,
    )

    summaries = []

    for strategy in [
        "BASE",
        "QUALITY",
    ]:

        selected = (
            trades.loc[
                trades[
                    "strategy"
                ]
                == strategy
            ]
            .sort_values(
                "exit_time"
            )
        )

        for side in [
            "ALL",
            "BUY",
            "SELL",
        ]:

            if side == "ALL":

                subset = (
                    selected
                )

            else:

                subset = (
                    selected.loc[
                        selected[
                            "direction"
                        ]
                        == side
                    ]
                )

            summaries.append(
                {
                    "strategy":
                        strategy,

                    "side":
                        side,

                    **strategy_stats(
                        subset[
                            "net_return"
                        ]
                    ),
                }
            )

    summary = (
        pd.DataFrame(
            summaries
        )
    )

    summary.to_csv(
        output_dir
        / "summary.csv",
        index=False,
    )

    if (
        importance_frames
    ):

        pd.concat(
            importance_frames
        ).to_csv(
            output_dir
            / "quality_importance.csv",
            index=False,
        )

    if (
        score_frames
    ):

        pd.concat(
            score_frames
        ).to_csv(
            output_dir
            / "test_scores.csv",
            index=False,
        )

    metadata[
        "status"
    ] = (
        "completed"
        if trade_frames
        else "no_evaluable_folds"
    )

    metadata[
        "evaluated_folds"
    ] = sum(
        r[
            "status"
        ]
        == "evaluated"
        for r
        in rows
    )

    save_metadata()

    return summary


# ============================================================
# 18. Yahoo Financeから5分足取得
# ============================================================

def download_snapshot(
    path
):

    import yfinance as yf

    now = (
        pd.Timestamp.now(
            tz="UTC"
        )
    )

    start = (
        now
        -
        pd.Timedelta(
            days=59
        )
    )

    print(
        "USD/JPY 5分足を取得しています..."
    )

    raw = (
        yf.download(
            "JPY=X",
            start=
                start.to_pydatetime(),
            end=
                now.to_pydatetime(),
            interval=
                "5m",
            auto_adjust=
                False,
            progress=
                False,
            threads=
                False,
            ignore_tz=
                False,
            keepna=
                True,
            multi_level_index=
                False,
            timeout=
                30,
        )
    )

    if (
        raw is None
        or
        raw.empty
    ):
        raise RuntimeError(
            "価格データを取得できませんでした。"
        )

    if isinstance(
        raw.columns,
        pd.MultiIndex,
    ):

        raw.columns = (
            raw.columns
            .get_level_values(0)
        )

    if (
        raw.index.tz
        is None
    ):

        raise ValueError(
            "取得データのタイムゾーンが不明です。"
        )

    raw = (
        raw[
            [
                "Open",
                "High",
                "Low",
                "Close",
            ]
        ]
    )

    missing = int(
        raw.isna()
        .any(axis=1)
        .sum()
    )

    raw = (
        raw.dropna()
    )

    incomplete = (
        raw.index
        .tz_convert(
            "UTC"
        )
        +
        pd.Timedelta(
            minutes=5
        )
        >
        now
    )

    incomplete_count = int(
        incomplete.sum()
    )

    raw = (
        raw.loc[
            ~incomplete
        ]
        .copy()
    )

    raw.index = (
        raw.index
        .tz_convert(
            "Asia/Tokyo"
        )
    )

    raw.to_csv(
        path,
        index_label=
            "timestamp",
    )

    load_bars(
        path
    )

    return {
        "source":
            "Yahoo Finance via yfinance",

        "symbol":
            "JPY=X",

        "interval":
            "5m",

        "yfinance_version":
            importlib.metadata.version(
                "yfinance"
            ),

        "requested_start_utc":
            str(start),

        "retrieved_at_utc":
            str(now),

        "missing_ohlc_rows_removed":
            missing,

        "incomplete_bars_removed":
            incomplete_count,

        "note":
            "New snapshot",
    }


# ============================================================
# 19. 診断
# ============================================================

def write_diagnostics(
    results
):

    summary = pd.read_csv(
        results
        / "summary.csv"
    )

    folds = pd.read_csv(
        results
        / "folds.csv"
    )

    trades = pd.read_csv(
        results
        / "trades.csv"
    )

    shown = (
        summary.copy()
    )

    for col in [
        "win_rate",
        "avg_return",
        "max_dd",
        "total_growth",
    ]:

        shown[
            col
        ] = (
            shown[
                col
            ]
            * 100
        )

    shown = shown.rename(
        columns={
            "win_rate":
                "win_rate_pct",

            "avg_return":
                "avg_return_pct",

            "max_dd":
                "max_dd_pct",

            "total_growth":
                "total_growth_pct",
        }
    )

    shown.to_csv(
        results
        / "summary_percent.csv",
        index=False,
    )

    print(
        "\n=============================="
    )

    print(
        "最終結果"
    )

    print(
        "=============================="
    )

    print(
        shown.to_string(
            index=False
        )
    )

    if trades.empty:

        print(
            "\n評価可能な取引がありません。"
        )

        print(
            folds.to_string(
                index=False
            )
        )

        return

    scores = pd.read_csv(
        results
        / "test_scores.csv"
    )

    trades[
        "signal_time"
    ] = pd.to_datetime(
        trades[
            "signal_time"
        ],
        utc=True,
    )

    scores[
        "signal_time"
    ] = pd.to_datetime(
        scores[
            "signal_time"
        ],
        utc=True,
    )

    scored = trades.merge(
        scores,
        on=[
            "fold",
            "signal_time",
        ],
        how="left",
        validate=
            "many_to_one",
    )

    if (
        scored[
            "p_quality"
        ].isna().any()
    ):
        raise ValueError(
            "取引とQualityスコアの対応に欠損があります。"
        )

    scored.to_csv(
        results
        / "trades_with_scores.csv",
        index=False,
    )

    base = (
        scored.loc[
            scored[
                "strategy"
            ]
            == "BASE"
        ]
        .copy()
    )

    edges = [
        0,
        0.40,
        0.50,
        0.55,
        0.60,
        0.65,
        0.70,
        np.nextafter(
            1.0,
            2.0,
        ),
    ]

    labels = [
        "[0,.40)",
        "[.40,.50)",
        "[.50,.55)",
        "[.55,.60)",
        "[.60,.65)",
        "[.65,.70)",
        "[.70,1]",
    ]

    base[
        "score_band"
    ] = pd.cut(
        base[
            "p_quality"
        ],
        edges,
        labels=
            labels,
        right=
            False,
    )

    band_rows = []

    fold_values = sorted(
        base[
            "fold"
        ]
        .unique()
        .tolist()
    )

    for fold_key in (
        [
            "ALL"
        ]
        +
        fold_values
    ):

        for side in [
            "ALL",
            "BUY",
            "SELL",
        ]:

            if (
                fold_key
                == "ALL"
            ):

                selected = (
                    base
                )

            else:

                selected = (
                    base.loc[
                        base[
                            "fold"
                        ]
                        == fold_key
                    ]
                )

            if side != "ALL":

                selected = (
                    selected.loc[
                        selected[
                            "direction"
                        ]
                        == side
                    ]
                )

            for band in (
                labels
            ):

                group = (
                    selected.loc[
                        selected[
                            "score_band"
                        ]
                        == band
                    ]
                    .sort_values(
                        "exit_time"
                    )
                )

                band_rows.append(
                    {
                        "fold":
                            fold_key,

                        "side":
                            side,

                        "score_band":
                            band,

                        "mean_quality_score":
                            group[
                                "p_quality"
                            ].mean(),

                        "small_sample_under_30":
                            len(group)
                            < 30,

                        **strategy_stats(
                            group[
                                "net_return"
                            ]
                        ),
                    }
                )

    bands = (
        pd.DataFrame(
            band_rows
        )
    )

    bands.to_csv(
        results
        / "quality_score_bands.csv",
        index=False,
    )

    # ----------------------------
    # コスト感度
    # ----------------------------

    stress_rows = []

    for strategy in [
        "BASE",
        "QUALITY",
    ]:

        selected = (
            trades.loc[
                trades[
                    "strategy"
                ]
                == strategy
            ]
            .sort_values(
                "exit_time"
            )
        )

        for multiplier in [
            1.0,
            1.5,
            2.0,
        ]:

            stressed = (
                selected[
                    "gross_return"
                ]
                -
                selected[
                    "cost"
                ]
                * multiplier
            )

            stress_rows.append(
                {
                    "strategy":
                        strategy,

                    "cost_multiplier":
                        multiplier,

                    **strategy_stats(
                        stressed
                    ),
                }
            )

    stress = (
        pd.DataFrame(
            stress_rows
        )
    )

    stress.to_csv(
        results
        / "cost_sensitivity.csv",
        index=False,
    )

    # ----------------------------
    # グラフ
    # ----------------------------

    import matplotlib

    matplotlib.use(
        "Agg"
    )

    import matplotlib.pyplot as plt

    trades[
        "entry_time"
    ] = pd.to_datetime(
        trades[
            "entry_time"
        ],
        utc=True,
    )

    trades[
        "exit_time"
    ] = pd.to_datetime(
        trades[
            "exit_time"
        ],
        utc=True,
    )

    first = (
        trades[
            "entry_time"
        ].min()
        -
        pd.Timedelta(
            seconds=1
        )
    )

    last = (
        trades[
            "exit_time"
        ].max()
    )

    equity_series = []

    for strategy in [
        "BASE",
        "QUALITY",
    ]:

        selected = (
            trades.loc[
                trades[
                    "strategy"
                ]
                == strategy
            ]
            .copy()
        )

        selected = (
            selected.sort_values(
                "exit_time"
            )
        )

        growth = pd.Series(
            (
                1
                +
                selected[
                    "net_return"
                ]
            ).cumprod()
            .to_numpy(),
            index=
                selected[
                    "exit_time"
                ],
            name=
                strategy,
        )

        growth = (
            growth.groupby(
                level=0
            )
            .last()
        )

        growth.loc[
            first
        ] = 1.0

        if (
            last
            not in growth.index
        ):

            growth.loc[
                last
            ] = (
                growth
                .sort_index()
                .iloc[-1]
            )

        equity_series.append(
            growth.sort_index()
        )

    equity = (
        pd.concat(
            equity_series,
            axis=1,
        )
        .sort_index()
        .ffill()
    )

    equity.to_csv(
        results
        / "equity.csv",
        index_label=
            "exit_time_utc",
    )

    fig, ax = plt.subplots(
        figsize=(
            10,
            4.5,
        )
    )

    for strategy in [
        "BASE",
        "QUALITY",
    ]:

        ax.step(
            equity.index,
            equity[
                strategy
            ],
            where=
                "post",
            label=
                strategy,
        )

    ax.set(
        title=
            "Out-of-sample closed-trade equity",

        ylabel=
            "Growth of 1",

        xlabel=
            "Exit time",
    )

    ax.legend()

    ax.grid(
        alpha=0.25
    )

    fig.autofmt_xdate()

    fig.tight_layout()

    fig.savefig(
        results
        / "equity.png",
        dpi=160,
    )

    plt.close(
        fig
    )

    totals = (
        summary.loc[
            summary[
                "side"
            ]
            == "ALL"
        ]
        .set_index(
            "strategy"
        )
    )

    if (
        "BASE"
        in totals.index
        and
        "QUALITY"
        in totals.index
    ):

        b = (
            totals.loc[
                "BASE"
            ]
        )

        q = (
            totals.loc[
                "QUALITY"
            ]
        )

        evaluated = (
            folds.loc[
                folds[
                    "status"
                ]
                == "evaluated"
            ]
        )

        improved = int(
            (
                evaluated[
                    "quality_avg_return"
                ]
                >
                evaluated[
                    "base_avg_return"
                ]
            ).sum()
        )

        reduction = (
            100
            *
            (
                1
                -
                q["trades"]
                /
                b["trades"]
            )
            if b[
                "trades"
            ]
            else np.nan
        )

        print(
            "\n=============================="
        )

        print(
            "Quality評価"
        )

        print(
            "=============================="
        )

        if (
            q[
                "avg_return"
            ]
            >
            b[
                "avg_return"
            ]
        ):

            print(
                "平均純損益は改善しています。"
            )

        else:

            print(
                "平均純損益は改善していません。"
            )

        print(
            "改善Fold:",
            improved,
            "/",
            len(
                evaluated
            ),
        )

        print(
            "QUALITY PF:",
            q[
                "profit_factor"
            ],
        )

        print(
            "取引削減率:",
            round(
                reduction,
                2,
            ),
            "%",
        )


# ============================================================
# 20. 画面表示＋ログ保存
# ============================================================

class Tee:
    """
    printを画面とログファイルの両方へ出す。
    """

    def __init__(
        self,
        screen,
        log,
    ):

        self.screen = (
            screen
        )

        self.log = (
            log
        )

    def write(
        self,
        text,
    ):

        self.screen.write(
            text
        )

        self.log.write(
            text
        )

        self.log.flush()

        return len(
            text
        )

    def flush(
        self
    ):

        self.screen.flush()

        self.log.flush()


# ============================================================
# 21. run()
# ============================================================

def run(
    csv_path=None,
    output_root=None,
):

    # Jupyterでは __file__ がないので
    # カレントディレクトリを使用する
    if (
        output_root
        is not None
    ):

        root = Path(
            output_root
        )

    else:

        root = (
            Path.cwd()
            / "fx_experiment_runs"
        )

    run_dir = (
        root
        /
        (
            datetime.now()
            .strftime(
                "%Y%m%d_%H%M%S"
            )
            +
            "_"
            +
            uuid.uuid4()
            .hex[:6]
        )
    )

    run_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    results = (
        run_dir
        / "results"
    )

    input_path = (
        run_dir
        / "usdjpy_5m.csv"
    )

    status = {
        "status":
            "running",

        "started_at_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

    error = None

    log_path = (
        run_dir
        / "console.log"
    )

    with log_path.open(
        "w",
        encoding="utf-8",
    ) as log:

        with contextlib.redirect_stdout(
            Tee(
                sys.stdout,
                log,
            )
        ):

            try:

                if (
                    csv_path
                    is None
                ):

                    status[
                        "input"
                    ] = (
                        download_snapshot(
                            input_path
                        )
                    )

                else:

                    bars = (
                        load_bars(
                            Path(
                                csv_path
                            )
                        )
                    )

                    bars.to_csv(
                        input_path,
                        index_label=
                            "timestamp",
                    )

                    status[
                        "input"
                    ] = {
                        "source":
                            "user-supplied CSV",

                        "original_filename":
                            Path(
                                csv_path
                            ).name,

                        "original_sha256":
                            hashlib.sha256(
                                Path(
                                    csv_path
                                ).read_bytes()
                            ).hexdigest(),
                    }

                print(
                    "固定保存したデータ:",
                    input_path,
                )

                print(
                    "5 foldの学習を開始します。"
                )

                print(
                    "Validation探索中は数分表示が止まる場合があります。"
                )

                run_experiment(
                    input_path,
                    results,
                )

                write_diagnostics(
                    results
                )

                status[
                    "status"
                ] = (
                    json.loads(
                        (
                            results
                            / "run.json"
                        ).read_text(
                            encoding="utf-8"
                        )
                    )[
                        "status"
                    ]
                )

            except Exception as exc:

                error = exc

                status[
                    "status"
                ] = "failed"

                status[
                    "error_type"
                ] = (
                    type(
                        exc
                    ).__name__
                )

                safe_trace = (
                    traceback
                    .format_exc()
                    .replace(
                        str(
                            Path.home()
                        ),
                        "[USER_HOME]",
                    )
                )

                (
                    run_dir
                    / "error.txt"
                ).write_text(
                    safe_trace,
                    encoding="utf-8",
                )

                print(
                    "停止しました:",
                    type(
                        exc
                    ).__name__,
                )

                print(
                    str(
                        exc
                    )
                )

    status[
        "finished_at_utc"
    ] = (
        datetime.now(
            timezone.utc
        ).isoformat()
    )

    (
        run_dir
        / "status.json"
    ).write_text(
        json.dumps(
            status,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    zip_path = (
        run_dir
        / "SEND_ME_results.zip"
    )

    with zipfile.ZipFile(
        zip_path,
        "w",
        zipfile.ZIP_DEFLATED,
    ) as archive:

        if results.exists():

            for path in sorted(
                results.rglob(
                    "*"
                )
            ):

                if path.is_file():

                    archive.write(
                        path,
                        path.relative_to(
                            run_dir
                        ),
                    )

        for filename in [
            "console.log",
            "status.json",
            "error.txt",
        ]:

            path = (
                run_dir
                / filename
            )

            if path.exists():

                archive.write(
                    path,
                    filename,
                )

    print(
        "\n=============================="
    )

    print(
        "完了"
    )

    print(
        "=============================="
    )

    print(
        "送ってほしいファイル:"
    )

    print(
        zip_path.resolve()
    )

    if (
        input_path.exists()
    ):

        print(
            "\n価格CSV:"
        )

        print(
            input_path.resolve()
        )

    if (
        error
        is not None
    ):

        print(
            "\nエラー詳細はZIP内のerror.txtに入っています。"
        )

    return zip_path


# ============================================================
# 22. コマンドライン実行
# ============================================================

def main():

    parser = (
        argparse.ArgumentParser(
            description=
                "USD/JPY BASE / QUALITY検証"
        )
    )

    parser.add_argument(
        "--csv",
        type=Path,
        help=
            "固定OHLC CSV。省略時は直近59日を取得。",
    )

    parser.add_argument(
        "--out-root",
        type=Path,
        help=
            "実行結果フォルダ",
    )

    # ========================================================
    # ここがJupyter対応の重要修正
    #
    # parse_args()ではなく
    # parse_known_args()を使う
    #
    # Jupyterが自動で渡す
    # -f kernel.json
    # を無視できる
    # ========================================================

    args, unknown = (
        parser.parse_known_args()
    )

    run(
        args.csv,
        args.out_root,
    )


# ============================================================
# 23. Jupyterで実行
# ============================================================

# Jupyterではmain()を使うと
# 内部引数の影響を受ける可能性があるため、
# 直接run()を呼ぶのが一番安全。

print(
    "コード読み込み完了"
)

print(
    "次のセルで run() を実行してください。"
)

## 元のセル index 19


In [ ]:
run()